In [ ]:
import os
import sys
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from pathlib import Path
import pickle

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
import seaborn as sns

# 1. Data Preparation

In [ ]:
%load_ext autoreload
%autoreload
import chip_utilities as utils

sys.path.insert(0, '..')
import sigmoid_fitting as sp
from experiment_data_loader import ExperimentDataLoader

exp_folder = "/Users/kautsarg/Documents/Final Project/Run Data/trial test data"
# exp_paths = [Path(exp_folder, name) for name in os.listdir(exp_folder) if name != ".DS_Store"]
exp_path = Path(exp_folder, "D20250808_E00_C00_F4500KHz_U_Sample_7")

data_path = os.path.join(exp_path, "curve_for_training.pkl")

with open(data_path, 'rb') as f:
    training_data = pickle.load(f)

timestamps = training_data["timestamps"]
Y_well = training_data["Y_well"]
dataset_name = training_data["dataset_name"]
dataset = training_data["dataset"]
kinetic_features = training_data["kinetic_features"]

In [ ]:
kinetic_features[0].head()

# 2. Additional Features Extraction

In [ ]:
for name, features_df, curves_2d in zip(dataset_name, kinetic_features, dataset):
    # FFI Extraction
    FFI = curves_2d[:,-1]
    
    # FFI - F0
    F_range = FFI - curves_2d[:,0]

    # temp group id
    
    # temp group id active pixels

    # different send n
    
    pass

# 3. Feature Linear Correlation Check

In [ ]:
for name, features_df in zip(dataset_name, kinetic_features):
    corr = features_df.corr()
    corr = corr.mask(corr.abs() < 0.5, 0)
    sns.heatmap(corr, cmap="coolwarm", xticklabels=True, yticklabels=True, linewidths=0.01, linecolor='grey')
    plt.title("name")
    plt.show()

# 4. Feature Importance (Non Linear Correlation) Check

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


for name, features_df in zip(dataset_name, kinetic_features):    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features_df)
    
    feature_cols = features_df.columns

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_scaled, Y_well)

    kinetic_importances_rf = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
    print(f"\n--- {name} Feature Importance ---")
    print(kinetic_importances_rf.sort_values(by='importance', ascending=False))